

Proposed Structure:


&ensp; \[Manual VS Automatic Curation\]

&ensp;&ensp; \[Processing Stage \]

&ensp;&ensp; \[File Format ( PDB VS EDM )\]


file Name: \{datasetID\}-{filename}

I.e. 



<details>
<summary> <b>ev2a</b>  </summary>


- 01-curated

    - 00-allformats

        - {datasetID} 

            ...
    
    - 01-dimple

        - 01-pdb
        
            - {datasetID} 

                ...
            
        - 02-edm

            - {datasetID} 

            ...

    - 02-pandda

        - 01-pdb
        
            - 01-input

            - 02-model

        - 02-mtz

            - 01-meanmap

            - 02-zmap

            - 03-events

    - 03-refine

        - 01-pdb

            - 01-ensemble ()

            - 02-overlay (refine)

            - 03-ground (split.ground)

            - 04-bound
        
        - 02-edm (refine.mtz or ccp4)

  
    - 04-depo


    - 05-ligand

        - 
    
- 02-automated

</details>

In [11]:
directoryFormat = { "01-curated": {
                        "00-allformats": [],
                        "01-dimple":{
                            "01-pdb":[],
                            "02-edm":[],
                        },
                        "02-pandda":{
                            "01-pdb":{
                                "01-input":[],
                                "02-model":[],
                                },
                            "02-edm":{
                                "01-meanmap":[],
                                "02-zmap":[],
                                "03-events":[],
                                },
                        },
                        "03-refine":{
                            "01-pdb":{
                                "01-ensemble":[],
                                "02-overlay":[],
                                "03-ground":[],
                                "04-bound":[],
                                },
                            "02-edm":[],
                        },
                        "04-depo":{
                            "01-pdb":[],
                            "02-edm":[],
                        },
                        "05-ligand":{
                            "01-pdb":[],
                            "02-edm":[],
                        },

                    },                        
                    "02-automated":[],                   
                }

In [ ]:
from pathlib import Path

rootPath = Path("/home/eoo22534/MyDB/xaidar/data/ev2a").mkdir( )

In [ ]:
def createWorkDir(rootDir: Path, dirFormat):
    for key, value in dirFormat.items():
        newPath = rootDir.joinpath(key)
        newPath.mkdir(parents=True, exist_ok=True)
        if isinstance(value, dict):
            createWorkDir(newPath, value)

In [ ]:
createWorkDir( rootPath, directoryFormat)

In [30]:
fileExtractFormat = { "01-curated": {
                        "00-allformats": [],
                        "01-dimple":{
                            "01-pdb":[],
                            "02-edm":[],
                        },
                        "02-pandda":{
                            "01-pdb":{
                                "01-input":["*pandda-input.pdb"],
                                "02-model":["*pandda-model.pdb"],
                                },
                            "02-edm":{
                                "01-meanmap":["*average-map.native.ccp4"],
                                "02-zmap":["*z_map.native.ccp4"],
                                "03-events":["*-event-*.ccp4", "*-event-*.mtz"],
                                },
                        },
                        "03-refine":{
                            "01-pdb":{
                                "01-ensemble":["*ensemble-model.pdb"],
                                "02-overlay": ["refine.pdb"],
                                "03-ground":["refine.split.ground-state.pdb"],
                                "04-bound":["refine.split.bound-state.pdb"],
                                },
                            "02-edm":["refine.ccp4", "refine.mtz"],
                        },
                        "04-depo":{
                            "01-pdb":[],
                            "02-edm":[],
                        },
                        "05-ligand":{
                            "01-pdb":[],
                            "02-edm":[],
                        },

                    },                        
                    "02-automated":[],                   
                }

In [19]:
print( list(rootPath.glob("*")))

[PosixPath('/home/eoo22534/MyDB/xaidar/data/ev2a/01-curated'), PosixPath('/home/eoo22534/MyDB/xaidar/data/ev2a/02-automated')]


In [28]:
import re
test = "A71EV2A-x0194-event_1_1-BDC_0.35_map.native.mtz"
test2 = "dimple.pdb"
dataset = "A71EV2A-x0194"[8:]

for file in [test, test2]:
    if  re.search( dataset, file):
        print( re.search( dataset, file).end() )
        print( file[re.search( dataset, file).end()+1:])


13
event_1_1-BDC_0.35_map.native.mtz


In [36]:
sourcePath = Path("/home/eoo22534/MyDB/xaidar/data/ev2a/01-curated/00-allformats")
rootPath = Path("/home/eoo22534/MyDB/xaidar/data/ev2a")
from shutil import copy2
import re

def cpFiles(sourcePath, targetPath, targetFiles):
    for patrnMatch in targetFiles:
        for dir in sourcePath.glob("*"):
            dataset = dir.name[8:]
            for file in dir.glob( patrnMatch):

                if  re.search( dataset, file.name):
                    fileName = file.name[re.search( dataset, file.name).end()+1:]
                else:
                    fileName = file.name

                newPath = targetPath.joinpath( f"{dataset}-{fileName}")
                copy2( file.as_posix(), newPath.as_posix())


def movFiles(rootDir, filesToExtract, sourcePath):
    for key, value in filesToExtract.items():
        newPath = rootDir.joinpath( key)

        if isinstance( value, dict):
            movFiles( newPath, filesToExtract[key], sourcePath)
        elif isinstance( value, list) and value != []:
            cpFiles( sourcePath, newPath, value)
            
movFiles(rootPath, fileExtractFormat,  sourcePath)